In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

In [3]:
# Load the training dataset as a NumPy array
# 1. Load the npz archive
train_file_path = "/home/mathilda/Documents/UCL/machine_learning/comp0245-final/robotnik_arm_task/data/proc/base_train_proc_20261101_1851.npz"
train_npz = np.load(train_file_path)
test_file_path = "/home/mathilda/Documents/UCL/machine_learning/comp0245-final/robotnik_arm_task/data/proc/base_test_proc_20261101_1851.npz"
test_npz = np.load(test_file_path)

# 2. Check what array names (keys) are stored inside
# print("Available arrays in file:", train_data.files)

# 3. Extract your specific arrays using their keys
# (Replace 'arr_0' or 'x' with the actual key names in your file)
train_data = train_npz["arr"]
test_data = test_npz["arr"]

# Optional: Close the file when done (or use a 'with' block below)
train_npz.close()
test_npz.close()

In [4]:
train_data
test_data

array([[ 9.00000000e+03, -3.68005305e-02, -3.05502414e-01, ...,
         3.00000000e+00,  3.00000000e+00, -3.00000000e+00],
       [ 9.00000000e+03, -3.68000832e-02, -3.05502414e-01, ...,
         3.00000000e+00,  3.00000000e+00, -3.00000000e+00],
       [ 9.00000000e+03, -3.67991885e-02, -3.05502305e-01, ...,
         3.00000000e+00,  3.00000000e+00, -3.00000000e+00],
       ...,
       [ 9.09900000e+03, -2.58838154e-02, -1.58446580e-01, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 9.09900000e+03, -2.58853252e-02, -1.58449316e-01, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 9.09900000e+03, -2.58815229e-02, -1.58448003e-01, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00]],
      shape=(1010078, 11))

In [22]:
X_train = train_data[:1000, 1:-4]  # All columns except the last four and the first
y_train = train_data[:1000, -4:]  # Last four columns
X_test = test_data[:1000, 1:-4]  # All columns except the last four and the first
y_test = test_data[:1000, -4:]  # Last four columns

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [23]:
X_train_tensor.shape, y_train_tensor.shape, X_test_tensor.shape, y_test_tensor.shape

(torch.Size([1000, 6]),
 torch.Size([1000, 4]),
 torch.Size([1000, 6]),
 torch.Size([1000, 4]))

In [24]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor) 
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [25]:
# Create data loaders
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False)

In [26]:
# Model Construction
class MLP(nn.Module):

  def __init__(self):
    super(MLP, self).__init__()
    self.flatten = nn.Flatten()
    self.hidden = nn.Linear(6, 16)
    self.relu = nn.ReLU()
    self.output = nn.Linear(16, 4)  # <-- 4 outputs for 4 continuous targets

  def forward(self, x):
    x = self.flatten(x)
    x = self.hidden(x)
    x = self.relu(x)
    x = self.output(x)
    return x


model = MLP()

In [27]:
# Model Compilation
criterion = nn.MSELoss()  # Mean Squared Error Loss for regression
# optimizer = optim.SGD(model.parameters(), lr=0.01)

optimizer = optim.Adam(model.parameters(), lr=0.01)

In [28]:
# 4. Model Training
epochs = 20
train_losses = []
valid_losses = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0

    for data, target in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)  # target is not one-hot encoded in PyTorch
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    # Calculate average loss for this epoch
    train_losses.append(epoch_loss / len(train_loader))

    print(f'Epoch {epoch+1}/{epochs}, Loss: {train_losses[-1]:.4f}')


Epoch 1/20, Loss: 3.8215
Epoch 2/20, Loss: 0.1325
Epoch 3/20, Loss: 0.0197
Epoch 4/20, Loss: 0.0137
Epoch 5/20, Loss: 0.0113
Epoch 6/20, Loss: 0.0096
Epoch 7/20, Loss: 0.0077
Epoch 8/20, Loss: 0.0058
Epoch 9/20, Loss: 0.0045
Epoch 10/20, Loss: 0.0035
Epoch 11/20, Loss: 0.0027
Epoch 12/20, Loss: 0.0021
Epoch 13/20, Loss: 0.0017
Epoch 14/20, Loss: 0.0014
Epoch 15/20, Loss: 0.0011
Epoch 16/20, Loss: 0.0010
Epoch 17/20, Loss: 0.0008
Epoch 18/20, Loss: 0.0007
Epoch 19/20, Loss: 0.0006
Epoch 20/20, Loss: 0.0006


In [29]:
# Model Evaluation
model.eval()
test_loss = 0
correct = 0

with torch.no_grad():
    for data, target in test_loader:
        output = model(data)
        test_loss += criterion(output, target).item()

test_loss /= len(test_loader)

print(f'Test Loss: {test_loss:.4f}')

Test Loss: 2.8385
